In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("Transactions.csv")

In [3]:
data.head()

,transaction_id,year,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2025,7:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2025,7:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2025,7:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2025,7:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2025,7:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


### Basic checking

In [4]:
data.shape

(149116, 11)

In [5]:
data.dtypes

transaction_id        int64
year                  int64
transaction_time     object
transaction_qty       int64
store_id              int64
store_location       object
product_id            int64
unit_price          float64
product_category     object
product_type         object
product_detail       object
dtype: object

In [6]:
data.isnull().sum()

transaction_id      0
year                0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
dtype: int64

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   year              149116 non-null  int64  
 2   transaction_time  149116 non-null  object 
 3   transaction_qty   149116 non-null  int64  
 4   store_id          149116 non-null  int64  
 5   store_location    149116 non-null  object 
 6   product_id        149116 non-null  int64  
 7   unit_price        149116 non-null  float64
 8   product_category  149116 non-null  object 
 9   product_type      149116 non-null  object 
 10  product_detail    149116 non-null  object 
dtypes: float64(1), int64(5), object(5)
memory usage: 12.5+ MB


In [8]:
data.describe()

,transaction_id,year,transaction_qty,store_id,product_id,unit_price
count,149116.000000,149116.0,149116.000000,149116.000000,149116.000000,149116.000000
mean,74737.371872,2025.0,1.438276,5.342063,47.918607,3.382219
std,43153.600016,0.0,0.542509,2.074241,17.930020,2.658723
min,1.000000,2025.0,1.000000,3.000000,1.000000,0.800000
25%,37335.750000,2025.0,1.000000,3.000000,33.000000,2.500000
50%,74727.500000,2025.0,1.000000,5.000000,47.000000,3.000000
75%,112094.250000,2025.0,2.000000,8.000000,60.000000,3.750000
max,149456.000000,2025.0,8.000000,8.000000,87.000000,45.000000


### Ensuring time format for "transaction_time"

In [9]:
data["transaction_time"] = pd.to_datetime(data["transaction_time"], format = "%H:%M:%S", errors = "coerce").dt.time

### Checking for invalid/duplicate measures

In [10]:
# invalid quantity (below 0) and invalid price (below 0)

invalid_quantity = data[data["transaction_qty"] <= 0]
invalid_price = data[data["unit_price"] <= 0]

print(f"Invalid quantity: {len(invalid_quantity)}")
print(f"Invalid price: {len(invalid_price)}")

Invalid quantity: 0
Invalid price: 0


In [11]:
# duplicate IDs

duplicates = data[data.duplicated(subset = ["transaction_id"])]
print(f"Duplicate transaction IDs: {len(duplicates)}")

Duplicate transaction IDs: 0


### Standardizing text columns

In [22]:
text_columns = ["product_category", "product_type", "product_detail", "store_location"]

for col in text_columns:
    data[col] = (data[col].str.strip().str.lower().str.replace(r"(?<!['’])\b[a-z]", lambda m: m.group(0).upper(), regex = True))

In [23]:
data["store_location"].unique()

array(['Lower Manhattan', "Hell's Kitchen", 'Astoria'], dtype=object)

### Derived column: revenue

In [13]:
data["revenue"] = data["transaction_qty"]*data["unit_price"]

In [14]:
data["revenue"].describe()

count    149116.000000
mean          4.686367
std           4.227099
min           0.800000
25%           3.000000
50%           3.750000
75%           6.000000
max         360.000000
Name: revenue, dtype: float64

In [15]:
data["revenue"].sum()

698812.3300000002

### Final checking

In [16]:
data.shape

(149116, 12)

In [17]:
data.isnull().sum()

transaction_id      0
year                0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
revenue             0
dtype: int64

In [24]:
data.to_csv("afficionado_cleaned.csv", index = False)

In [28]:
product_revenue = data.groupby("product_detail")["revenue"].sum().reset_index()
product_revenue = product_revenue.sort_values("revenue", ascending = False)
product_revenue["Cumulative Revenue"] = product_revenue["revenue"].cumsum()
product_revenue["Cumulative PCT"] = product_revenue["Cumulative Revenue"]/product_revenue["revenue"].sum()*100
product_revenue["Rank"] = range(1, len(product_revenue) + 1)

product_revenue.to_csv("pareto_data.csv", index = False)